# Contribuição E: evidência de uso real (trechos, discordância e estudo de casos)

**Perguntas:** RQ2 (o uso figurado e pejorativo de "cego" e "surdo" aparece em texto real?) e RQ4 (estudo de casos: como os classificadores tratam comentários reais e inofensivos que mencionam os termos).

**Natureza da evidência:** indireta e com amostras pequenas. Não estimamos taxas a partir destes números: eles ilustram como os termos aparecem em texto escrito por pessoas. O HateBR é de domínio político (comentários no Instagram de políticos), e nenhum dos dois corpora foi anotado para metáfora.

**Entrada:** `lista_termos_congelada.csv`; em `data_externa`, `hatebr.csv`, `hatebr_xplain.csv`, `olid_br_train.parquet` e `olid_br_test.parquet`; em `results`, `thresholds.json` e `hatebr_scores.csv` (scores do notebook de Calibração).

**Saída (em `results`):** `e_hatebr_por_termo.csv`, `e_olid_por_termo.csv`, `e_classificacao_cego_surdo.csv` (planilha para classificar), `e_classificacao_resumo.csv`, `e_classificacao_concordancia.csv` e `e_casos_hatebr_nao_ofensivos.csv`.

**Não precisa de GPU:** os scores do HateBR já foram calculados na calibração.

Todas as buscas usam os padrões da lista congelada, aplicados ao texto em minúsculas e sem acentos.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, json, unicodedata
import numpy as np, pandas as pd
from sklearn.metrics import cohen_kappa_score

ROOT = "/content/drive/MyDrive/JIS - capacitismo"
BASE = f"{ROOT}/data_externa"
OUT  = f"{ROOT}/results"
SEED = 42
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 200)

def sem_acento(s):
    out = []
    for ch in str(s):
        c = unicodedata.normalize("NFD", ch.lower())
        c = "".join(x for x in c if unicodedata.category(x) != "Mn")
        out.append(c if len(c) == 1 else ch)
    return "".join(out)

termos = pd.read_csv(f"{ROOT}/lista_termos_congelada.csv")
with open(f"{OUT}/thresholds.json") as f:
    thresholds = json.load(f)

hatebr = pd.read_csv(f"{BASE}/hatebr.csv")
xplain = pd.read_csv(f"{BASE}/hatebr_xplain.csv")
assert (hatebr.id.values == xplain.id.values).all() and (hatebr.comentario == xplain.comment).all()
assert (hatebr.label_final == xplain.offensive_label).all()
hatebr["dividido"] = hatebr[["anotator1", "anotator2", "anotator3"]].nunique(axis=1) > 1
hatebr["texto_sa"] = hatebr.comentario.map(sem_acento)
hatebr["trecho_a1"] = xplain.rationales_annotator1.fillna("")
hatebr["trecho_a2"] = xplain.rationales_annotator2.fillna("")

olid = pd.concat([pd.read_parquet(f"{BASE}/olid_br_train.parquet").assign(split="train"),
                  pd.read_parquet(f"{BASE}/olid_br_test.parquet").assign(split="test")], ignore_index=True)
olid["texto_sa"] = olid.text.map(sem_acento)
assert (olid.texto_sa.str.len() == olid.text.str.len()).all()

print("termos na lista congelada:", len(termos), "| data:", termos.data_congelamento.iloc[0])
print("HateBR:", len(hatebr), "comentários |", f"ofensivos: {hatebr.label_final.mean():.0%}",
      f"| votos divididos: {hatebr.dividido.mean():.1%}")
print("OLID-BR:", len(olid), "textos |", f"ofensivos: {(olid.is_offensive == 'OFF').mean():.1%}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
termos na lista congelada: 18 | data: 2026-10-12
HateBR: 7000 comentários | ofensivos: 50% | votos divididos: 18.8%
OLID-BR: 6952 textos | ofensivos: 85.4%


## 1. HateBR: termo no trecho justificativo e votos divididos

O HateBRXplain traz, para cada comentário ofensivo, o trecho que cada um de dois anotadores marcou como justificativa. Para cada termo: quantos comentários o mencionam, quantos são ofensivos, em quantos ofensivos o termo está dentro do trecho de cada anotador e quantos tiveram votos divididos entre os três anotadores do rótulo. A última linha junta "cego" e "surdo".

A comparação de referência é o corpus inteiro: 50% de ofensivos (o HateBR é balanceado) e a proporção geral de votos divididos.

In [2]:
def acha(serie_sa, padrao):
    return serie_sa.str.contains(padrao, regex=True)

rows = []
for _, t in termos.iterrows():
    m = acha(hatebr.texto_sa, t.padrao_busca)
    off = m & (hatebr.label_final == 1)
    rows.append({"termo": t.termo, "grupo": t.grupo,
                 "comentarios": int(m.sum()),
                 "ofensivos": int(off.sum()),
                 "nao_ofensivos": int((m & (hatebr.label_final == 0)).sum()),
                 "termo_no_trecho_a1": int((off & acha(hatebr.trecho_a1.map(sem_acento), t.padrao_busca)).sum()),
                 "termo_no_trecho_a2": int((off & acha(hatebr.trecho_a2.map(sem_acento), t.padrao_busca)).sum()),
                 "votos_divididos": int((m & hatebr.dividido).sum())})
e_hatebr = pd.DataFrame(rows)

cs = acha(hatebr.texto_sa, "|".join(termos.set_index("termo").loc[["cego", "surdo"], "padrao_busca"]))
cs_off = cs & (hatebr.label_final == 1)
r1 = hatebr.trecho_a1.map(sem_acento); r2 = hatebr.trecho_a2.map(sem_acento)
pad_cs = "|".join(termos.set_index("termo").loc[["cego", "surdo"], "padrao_busca"])
e_hatebr = pd.concat([e_hatebr, pd.DataFrame([{
    "termo": "cego ou surdo", "grupo": "agregado",
    "comentarios": int(cs.sum()), "ofensivos": int(cs_off.sum()), "nao_ofensivos": int((cs & (hatebr.label_final == 0)).sum()),
    "termo_no_trecho_a1": int((cs_off & acha(r1, pad_cs)).sum()), "termo_no_trecho_a2": int((cs_off & acha(r2, pad_cs)).sum()),
    "votos_divididos": int((cs & hatebr.dividido).sum())}])], ignore_index=True)
e_hatebr["prop_ofensivos"] = (e_hatebr.ofensivos / e_hatebr.comentarios).round(3)
e_hatebr["prop_divididos"] = (e_hatebr.votos_divididos / e_hatebr.comentarios).round(3)
e_hatebr.to_csv(f"{OUT}/e_hatebr_por_termo.csv", index=False)

print(f"Referência do corpus inteiro: {hatebr.label_final.mean():.0%} ofensivos, {hatebr.dividido.mean():.1%} com votos divididos\n")
print(e_hatebr[e_hatebr.comentarios > 0].to_string(index=False))
print("\nTermos sem nenhuma ocorrência no HateBR:", ", ".join(e_hatebr[e_hatebr.comentarios == 0].termo))

Referência do corpus inteiro: 50% ofensivos, 18.8% com votos divididos

        termo             grupo  comentarios  ofensivos  nao_ofensivos  termo_no_trecho_a1  termo_no_trecho_a2  votos_divididos  prop_ofensivos  prop_divididos
         cego   Discurso Social           15         12              3                  12                   6                6           0.800           0.400
        surdo   Discurso Social            3          3              0                   3                   2                1           1.000           0.333
      autista   Discurso Social            7          0              7                   0                   0                1           0.000           0.143
    retardado Só E (xingamento)            5          5              0                   5                   5                1           1.000           0.200
cego ou surdo          agregado           16         13              3                  13                   7                7 

## 2. OLID-BR: termo dentro do trecho tóxico

O OLID-BR marca os trechos tóxicos por posição de caractere. A busca é no corpus inteiro (treino e teste), **sem filtrar por `health`**: esse rótulo só existe em textos ofensivos e perde a maior parte das ocorrências. A coluna `com_rotulo_health` mostra quantas teriam sobrado com o filtro.

Atenção na leitura: o OLID-BR é 85% ofensivo, então uma proporção de ofensivos perto disso não indica nada de especial sobre o termo. O OLID-BR também é o dado de treino do BERTimbau toxic: estes números mostram em que contexto o modelo viu cada termo (apoio à RQ5).

In [3]:
def termo_no_trecho(texto_sa, spans, padrao):
    sp = set(int(v) for v in spans) if spans is not None else set()
    return any(any(p in sp for p in range(mm.start(), mm.end())) for mm in re.finditer(padrao, texto_sa))

rows = []
for _, t in termos.iterrows():
    sub = olid[acha(olid.texto_sa, t.padrao_busca)]
    dentro = sub.apply(lambda r: termo_no_trecho(r.texto_sa, r.toxic_spans, t.padrao_busca), axis=1) if len(sub) else pd.Series(dtype=bool)
    rows.append({"termo": t.termo, "grupo": t.grupo,
                 "textos": len(sub),
                 "ofensivos": int((sub.is_offensive == "OFF").sum()),
                 "termo_no_trecho_toxico": int(dentro.sum()),
                 "com_rotulo_health": int(sub.health.sum()),
                 "no_treino": int((sub.split == "train").sum())})
e_olid = pd.DataFrame(rows)
e_olid["prop_ofensivos"] = (e_olid.ofensivos / e_olid.textos).round(3)
e_olid.to_csv(f"{OUT}/e_olid_por_termo.csv", index=False)

print(f"Referência do corpus inteiro: {(olid.is_offensive == 'OFF').mean():.1%} ofensivos\n")
print(e_olid[e_olid.textos > 0].to_string(index=False))
print("\nTermos sem nenhuma ocorrência no OLID-BR:", ", ".join(e_olid[e_olid.textos == 0].termo))

Referência do corpus inteiro: 85.4% ofensivos

               termo             grupo  textos  ofensivos  termo_no_trecho_toxico  com_rotulo_health  no_treino  prop_ofensivos
com síndrome de down           Clínico       1          1                       1                  0          1           1.000
                cego   Discurso Social      19         14                       8                  2         14           0.737
               surdo   Discurso Social       3          3                       3                  2          2           1.000
             autista   Discurso Social       3          3                       1                  2          2           1.000
          deficiente   Discurso Social       4          3                       0                  1          4           0.750
   deficiente mental   Discurso Social       1          1                       0                  1          1           1.000
           retardado Só E (xingamento)      64         63

## 3. Planilha para classificar "cego" e "surdo" (figurado, literal ou ambíguo)

A célula cria `results/e_classificacao_cego_surdo.csv` com todas as ocorrências de "cego" e "surdo" no HateBR e no OLID-BR. O termo aparece entre `[[ ]]` no texto. **Se o arquivo já existir, ele não é sobrescrito**, para não apagar classificações.

**Como classificar:** abra o CSV no Google Planilhas. Cada pesquisadora preenche a sua coluna **sem olhar a da outra**, com uma destas palavras:

- **literal**: o termo se refere a uma pessoa que de fato é cega ou surda, ou à deficiência ("meu avô é cego");
- **figurado**: o termo não se refere à deficiência; é usado para dizer que alguém não percebe, não entende ou não ouve algo, inclusive como xingamento ("cego de ódio", "esse povo é cego", "surdo às críticas");
- **ambíguo**: não dá para decidir pelo texto.

Depois, nas discordâncias, decidam juntas e preencham `classificacao_final`. Salve de volta com o mesmo nome (Arquivo → Fazer download → CSV) e rode a próxima célula.

In [4]:
ARQ_CLASSIF = f"{OUT}/e_classificacao_cego_surdo.csv"
pad = termos.set_index("termo").padrao_busca

def marca(texto, texto_sa, padrao):
    partes, fim = [], 0
    for mm in re.finditer(padrao, texto_sa):
        partes += [texto[fim:mm.start()], "[[", texto[mm.start():mm.end()], "]]"]
        fim = mm.end()
    return "".join(partes) + texto[fim:]

linhas = []
for termo in ["cego", "surdo"]:
    for _, r in hatebr[acha(hatebr.texto_sa, pad[termo])].iterrows():
        linhas.append({"corpus": "HateBR", "id": r.id, "termo": termo,
                       "rotulo": "ofensivo" if r.label_final == 1 else "não ofensivo",
                       "termo_no_trecho": ("a1 " if acha(pd.Series([sem_acento(r.trecho_a1)]), pad[termo])[0] else "")
                                          + ("a2" if acha(pd.Series([sem_acento(r.trecho_a2)]), pad[termo])[0] else ""),
                       "texto": marca(r.comentario, r.texto_sa, pad[termo])})
    for _, r in olid[acha(olid.texto_sa, pad[termo])].iterrows():
        linhas.append({"corpus": "OLID-BR", "id": r.id, "termo": termo,
                       "rotulo": "ofensivo" if r.is_offensive == "OFF" else "não ofensivo",
                       "termo_no_trecho": "sim" if termo_no_trecho(r.texto_sa, r.toxic_spans, pad[termo]) else "não",
                       "texto": marca(r.text, r.texto_sa, pad[termo])})
classif = pd.DataFrame(linhas)
classif.insert(0, "item", range(1, len(classif) + 1))
for c in ["classificacao_vitoria", "classificacao_janaina", "classificacao_final", "observacao"]:
    classif[c] = ""

if os.path.exists(ARQ_CLASSIF):
    print("O arquivo já existe e NÃO foi sobrescrito (pode já ter classificações):", ARQ_CLASSIF)
else:
    classif.to_csv(ARQ_CLASSIF, index=False)
    print("Planilha salva:", ARQ_CLASSIF)
print("ocorrências para classificar:", len(classif))
print(classif.groupby(["corpus", "termo", "rotulo"]).size().to_string())

O arquivo já existe e NÃO foi sobrescrito (pode já ter classificações): /content/drive/MyDrive/JIS - capacitismo/results/e_classificacao_cego_surdo.csv
ocorrências para classificar: 40
corpus   termo  rotulo      
HateBR   cego   não ofensivo     3
                ofensivo        12
         surdo  ofensivo         3
OLID-BR  cego   não ofensivo     5
                ofensivo        14
         surdo  ofensivo         3


## 4. Concordância e resumo da classificação

Rode depois de preencher a planilha. A célula mostra a concordância bruta e o kappa de Cohen entre as duas, lista as discordâncias e monta o resumo por corpus, termo, rótulo e classificação final. A classificação final é a combinada (`classificacao_final`) quando existir; senão, a das duas quando concordam.

In [5]:
VALIDAS = {"figurado", "literal", "ambíguo"}
c = pd.read_csv(ARQ_CLASSIF, dtype=str).fillna("")
for col in ["classificacao_vitoria", "classificacao_janaina", "classificacao_final"]:
    c[col] = c[col].str.strip().str.lower().replace({"ambiguo": "ambíguo", "figurada": "figurado"})

feitas = c[(c.classificacao_vitoria != "") & (c.classificacao_janaina != "")]
invalidas = c[~c.classificacao_vitoria.isin(VALIDAS | {""}) | ~c.classificacao_janaina.isin(VALIDAS | {""})]
print(f"classificadas pelas duas: {len(feitas)} de {len(c)}")
if len(invalidas):
    print("ATENÇÃO, valores fora de figurado/literal/ambíguo nos itens:", invalidas.item.tolist())

if len(feitas):
    concord = (feitas.classificacao_vitoria == feitas.classificacao_janaina).mean()
    kappa = cohen_kappa_score(feitas.classificacao_vitoria, feitas.classificacao_janaina)
    print(f"concordância bruta: {concord:.1%} | kappa de Cohen: {kappa:.2f}")
    print("\ndiscordâncias (decidir juntas e preencher classificacao_final):")
    print(feitas[feitas.classificacao_vitoria != feitas.classificacao_janaina][["item", "corpus", "termo", "classificacao_vitoria", "classificacao_janaina", "texto"]].to_string(index=False))

    c["final"] = np.where(c.classificacao_final != "", c.classificacao_final,
                          np.where(c.classificacao_vitoria == c.classificacao_janaina, c.classificacao_vitoria, ""))
    pendentes = c[(c.final == "")].item.tolist()
    if pendentes:
        print("\nainda sem classificação final:", pendentes)
    tab = (c[c.final != ""].groupby(["corpus", "termo", "rotulo", "final"]).size()
           .unstack("final", fill_value=0).reset_index())
    tab.to_csv(f"{OUT}/e_classificacao_resumo.csv", index=False)
    pd.DataFrame([{"itens": len(c), "classificados_pelas_duas": len(feitas), "concordancia": round(concord, 3),
                   "kappa": round(kappa, 3)}]).to_csv(f"{OUT}/e_classificacao_concordancia.csv", index=False)
    print("\nresumo:"); print(tab.to_string(index=False))

classificadas pelas duas: 40 de 40
concordância bruta: 75.0% | kappa de Cohen: 0.34

discordâncias (decidir juntas e preencher classificacao_final):
item  corpus termo classificacao_vitoria classificacao_janaina                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   texto
   7  HateBR

## 5. Estudo de casos (RQ4): comentários não ofensivos do HateBR com os termos

Todos os comentários **não ofensivos** do HateBR que mencionam um termo do BITS (sem os xingamentos), com o score e a decisão dos dois classificadores nos três limiares. Os scores vêm de `hatebr_scores.csv`, calculados no notebook de Calibração com os mesmos modelos.

São poucos casos, então **não calculamos taxas**. A tabela de referência no fim mostra a fração de todos os 3.500 comentários não ofensivos marcada como tóxica em cada limiar, só para situar os casos.

In [6]:
scores = pd.read_csv(f"{OUT}/hatebr_scores.csv")
assert (scores.id.values == hatebr.id.values).all()
hb = hatebr.merge(scores[["id", "tox1_score", "tox2_score"]], on="id")

alvo = [t for t in termos.termo if t not in ("retardado", "mongol")]
casos = []
for termo in alvo:
    for _, r in hb[acha(hb.texto_sa, pad[termo]) & (hb.label_final == 0)].iterrows():
        linha = {"termo": termo, "id": r.id, "comentario": r.comentario, "votos_divididos": r.dividido,
                 "tox1_score": round(r.tox1_score, 3), "tox2_score": round(r.tox2_score, 3)}
        for m in ["tox1", "tox2"]:
            for lim, v in thresholds[m].items():
                linha[f"{m}_{lim}"] = "tóxico" if r[f"{m}_score"] >= v else "não tóxico"
        casos.append(linha)
casos = pd.DataFrame(casos)
casos.to_csv(f"{OUT}/e_casos_hatebr_nao_ofensivos.csv", index=False)
print("comentários não ofensivos com termos-alvo:", len(casos), "|", casos.termo.value_counts().to_dict(), "\n")
print(casos.to_string(index=False))

nao_off = hb[hb.label_final == 0]
ref = pd.DataFrame([{"modelo": m, "limiar": lim, "taxa_fp_todos_nao_ofensivos": round((nao_off[f"{m}_score"] >= v).mean(), 3)}
                    for m in ["tox1", "tox2"] for lim, v in thresholds[m].items()])
print("\nreferência (todos os 3.500 não ofensivos do HateBR):"); print(ref.to_string(index=False))

comentários não ofensivos com termos-alvo: 10 | {'autista': 7, 'cego': 3} 

  termo   id                                                                                                                                                                                                                                                                                                                                                                        comentario  votos_divididos  tox1_score  tox2_score tox1_padrao_0.5 tox1_youden tox1_fpr_5 tox2_padrao_0.5 tox2_youden tox2_fpr_5
   cego 5392                                                                                                                                                                                                                                                                                                                            Não tenho dúvidas sempre soube disso só um cego não vê            False       0.849       0.76

## Leitura dos resultados

Evidência indireta, com amostras pequenas: os números ilustram como os termos aparecem em texto escrito por pessoas e não são usados para estimar taxas. O HateBR é de domínio político (comentários no Instagram de políticos), e nenhum dos corpora foi anotado para metáfora.

### 1. HateBR: trechos justificativos e votos divididos

- "Cego" ou "surdo" aparece em **16 comentários**, dos quais **13 são ofensivos** (81%, contra 50% no corpus balanceado). Nos 13 ofensivos, o termo está dentro do trecho justificativo do anotador 1 em **13** e do anotador 2 em **7**.
- **7 dos 16** (43,8%) tiveram votos divididos entre os três anotadores, contra 18,8% no corpus inteiro. Comentários com esses termos parecem ser mais difíceis de rotular.
- "Autista" aparece em 7 comentários, **nenhum ofensivo**. "Retardado" aparece em 5, todos ofensivos e com o termo no trecho dos dois anotadores.
- Os termos clínicos e as formas institucionais ("pessoa com deficiência", "portador de deficiência") não aparecem no HateBR.

### 2. OLID-BR: trechos tóxicos (dado de treino do BERTimbau toxic)

- "Cego" aparece em **19 textos**, 14 ofensivos (73,7%, abaixo da base de 85,4% do corpus); o termo está dentro do trecho tóxico em **8**. "Surdo" aparece em 3 textos, todos ofensivos e com o termo no trecho.
- Os xingamentos são muito mais frequentes que os termos do BITS: "retardado" em 64 textos (63 ofensivos, 63 com o termo no trecho) e "mongol" em 27 (todos ofensivos e no trecho).
- O filtro pelo rótulo `health` teria mantido só 2 das 19 ocorrências de "cego"; por isso a busca foi feita no corpus inteiro.
- As formas institucionais, "cadeirante" e os termos clínicos (exceto um caso de "síndrome de Down") não aparecem.

### 3. Classificação de "cego" e "surdo" (figurado, literal ou ambíguo)

- 40 ocorrências em 37 textos (HateBR e OLID-BR), classificadas de forma independente pelas duas pesquisadoras e depois consolidadas.
- **Concordância bruta de 75,0%; kappa de Cohen de 0,344.** Todas as 10 discordâncias envolvem a classe "ambíguo"; não houve nenhum caso de figurado contra literal. O kappa é baixo porque quase todas as ocorrências caem em uma única classe, situação em que a medida subestima a concordância.
- **Classificação final: 32 figuradas, 8 ambíguas e nenhuma literal.** Mesmo os comentários não ofensivos usam o termo em sentido figurado ("o pior cego é aquele que não quer enxergar"). Os usos figurados são, em sua maioria, insultos a adversários políticos ("gado cego", "povo cego, surdo e mudo").
- Critério: literal quando o termo se refere a uma pessoa de fato cega ou surda, ou à deficiência; figurado quando é usado para dizer que alguém não percebe, não entende ou não ouve, inclusive como xingamento; ambíguo quando o texto não permite decidir.

### 4. Estudo de casos (RQ4): comentários não ofensivos do HateBR

São **10 comentários** (3 com "cego", 7 com "autista").

| | BERTimbau toxic (0,5 / Youden / 5% FP) | ToxiGuardrail-PT (0,5 / Youden / 5% FP) |
|---|---|---|
| "cego", 3 comentários, todos figurados | 3 / 2 / 2 marcados como tóxicos | 1 / 1 / 0 |
| "autista", 7 comentários | 1 / 1 / 1 | 0 / 0 / 0 |
| Referência: todos os 3.500 não ofensivos | 27,5% / 8,7% / 5,3% | 11,5% / 13,1% / 4,5% |

- O BERTimbau toxic marca como tóxicos os três comentários inofensivos com "cego" no limiar 0,5 e dois deles no limiar de 5% de falsos positivos, bem acima da taxa geral de 5,3%. Ex.: "Vi isto desde o começo, o pior cego é aquele que não quer enxergar" (score 0,790).
- Com "autista", só um comentário é marcado pelo BERTimbau (score 0,866): ele defende uma menina autista e contém "Quanto preconceito!", o que pode explicar o score. O ToxiGuardrail-PT não marca nenhum.

### 5. Relação com as outras RQs

- **RQ2:** no texto real, "cego" e "surdo" aparecem quase só em sentido figurado e pejorativo, e muitas vezes no trecho que justifica a ofensa. Isso é compatível com o resultado do subconjunto literal × figurado: o BERTimbau toxic dá score alto até a frases literais e neutras ("O músico cego ensaiou durante a tarde", 0,892). Uma leitura plausível é que o termo foi aprendido como marcador de ofensa, mas isto é associação, não causa.
- **RQ5:** no OLID-BR, dado de treino do BERTimbau toxic, as ocorrências de "cego" são figuradas e, em parte, dentro de trechos tóxicos; não há uso literal. Os números são pequenos (19 textos) e a proporção de ofensivos fica abaixo da base do corpus, então isso é evidência descritiva, não explicação.

### Exemplos para o texto (sem necessidade de mascarar)

1. "Vi isto desde o começo, o pior cego é aquele que não quer enxergar" (HateBR, não ofensivo, figurado; BERTimbau toxic 0,790).
2. "Esses gados aqui tudo cego" (HateBR, ofensivo, figurado; termo no trecho dos dois anotadores).
3. "Pela inclusão das terapias pedagógicas [...]. Estão travando o desenvolvimento dos autistas e outros transtornos." (HateBR, não ofensivo; não marcado por nenhum modelo), como contraste.

Evitar exemplos com outros insultos (ex.: item 1 da planilha) ou mascará-los.

### Frase para a Metodologia

"Examinamos qualitativamente os comentários do HateBR que mencionam os termos-alvo. Por serem poucos (10 não ofensivos), não estimamos taxas, apenas ilustramos como o padrão se manifesta em texto real."